# Checkpoint별 키워드 정확도 평가
모든 checkpoint에 대해 test 데이터로 키워드(importance, department, complaint_type, emotion) 정확도를 측정하고 JSON으로 저장합니다.

## ▶ 수정 변수 (여기서만 수정하면 됩니다)

In [ ]:
# ===================== 수정 변수 =====================
HF_TOKEN        = ""                                          # HuggingFace read-only token
DATASET_ID      = "leejunho12316/seoul-mayor-hope-labeled-backup2500"  # HuggingFace 데이터셋 ID
TEST_START_IDX  = 2000                                        # test 데이터 시작 인덱스

BASE_MODEL_ID   = "Qwen/Qwen2.5-0.5B-Instruct"               # tokenizer 로드용 베이스 모델 ID
MODEL_NAME      = "qwen2.5-0.5b"                              # checkpoint 폴더명 (output_dir)
CHECKPOINT_DIR  = "3.Fine_Tuning/qwen2.5-0.5b"               # checkpoint들이 저장된 디렉토리
OUTPUT_JSON     = "3.Fine_Tuning/full_results_checkpoints.json"  # 결과 저장 경로

BATCH_SIZE      = 16     # 추론 배치 크기
MAX_NEW_TOKENS  = 1024   # 최대 생성 토큰 수
# =====================================================

## 0. 패키지 설치

In [ ]:
%pip install -q "torch==2.4.0"
%pip install -q "transformers==4.45.1" "datasets==3.0.1" "accelerate==0.34.2" "trl==0.11.1" "peft==0.13.0"
%pip install -q langchain-core pydantic tqdm

## 1. Import

In [ ]:
import os
import json
import ast
import re
import random

import torch
from datasets import load_dataset, Dataset
from transformers import AutoTokenizer, pipeline
from peft import AutoPeftModelForCausalLM
from huggingface_hub import login
from tqdm import tqdm
from langchain_core.output_parsers import JsonOutputParser
from pydantic import BaseModel, Field

## 2. 데이터 로드 및 test 데이터 준비

In [ ]:
login(token=HF_TOKEN)

In [ ]:
dataset    = load_dataset(DATASET_ID)
df_original = dataset['train'].to_pandas()
df_original = df_original[df_original['assistant'].notna()].reset_index(drop=True)

test_df = df_original[TEST_START_IDX:].copy()
print(f"전체 데이터: {len(df_original)}행 / test 데이터: {len(test_df)}행")

## 3. System Prompt 및 데이터 포맷

In [ ]:
def get_system_prompt():
    importance_items = [
        ("높음", "행정적 조치, 전문적인 도움이 필요한 글. 특정한 문제가 발생했거나 부당한 처우에 대한 항의."),
        ("보통", "보통의 의견이나 제안, 생각을 담은 글. 소식, 칭찬, 정보를 담은 글 등."),
        ("낮음", "감정적으로만 작성한 글. 어그로성 글. 특정 개인에 대한 근거 없고 맹목적인 비난 글. 비논리적이고 문맥에 일관성이 없는 글. 작성이 온전히 다 되지 않은 글. 등")
    ]
    random.shuffle(importance_items)
    importance_str = "\n".join([f"- {l} : {d}" for l, d in importance_items])

    department_items = [
        ("교통실", "버스·지하철·택시, 대중교통 정책, 자전거·킥보드·보행, 주차, 신호, 불법주정차, 한강버스, 교통카드, 도로교통, 자율주행"),
        ("복지실", "기초생활보장, 저소득층 지원, 노숙인, 어르신 돌봄, 장애인 지원, 아동·청소년 복지, 한부모·다문화가족, 중장년 지원"),
        ("경제실", "창업·스타트업, 소상공인·전통시장 지원, 청년 취업·일자리, 중소기업 자금, 소비자 권익, 생활임금·노동정책, 자영업자, 지원금"),
        ("기후환경본부", "쓰레기·재활용, 소각장, 미세먼지·대기질, 동물보호, 탄소중립·신재생에너지, 친환경차·전기차 충전, 도시공원, 식품안전"),
        ("문화본부", "도서관, 박물관·문화시설, 공연·예술 지원, 문화유산, 전통문화, 관광 계획, 공원 시설 관리/조성"),
        ("시민건강국", "보건소, 응급의료, 감염병·방역, 정신건강, 예방접종, 치매 예방, 공중위생, 건강증진, 마약 대응, 금연 지원, 금연구역 관리"),
        ("재난안전실", "재난대응, 취약시설 점검, 도로·보도 안전, 대피소, 시민안전보험, 제설, 인파 안전관리, 도로공사 안전 관리, 공사현장 관리"),
        ("주택실", "재개발·재건축, 공공주택, 전세사기, 건축인허가, 도시계획, 주거환경개선, 도시재생, 시설물 관리, 공공시설 관리, 부동산, 사유지"),
        ("여성가족실", "보육·어린이집, 저출생 대응, 아동학대 예방, 청소년 지원·보호, 성폭력·성희롱 예방, 디지털성범죄, 여성 안전, 양성평등"),
        ("분류 보류", "정부 부서 관할 이외의 기관에 대한 내용. 정치적인 내용."),
    ]
    random.shuffle(department_items)
    department_str = "\n".join([f"- {l} : {d}" for l, d in department_items])

    complaint_type_items = [
        ("신고", "불법 행위, 위험 상황, 규정 위반 등 제3자나 시설에 대한 문제를 알리는 경우"),
        ("문의", "제도, 정책, 절차, 방법 등에 대한 정보나 안내를 요청하는 경우"),
        ("건의", "정책 개선, 시설 설치, 제도 변경 등을 제안하는 경우"),
        ("항의", "행정 처리나 처우에 대한 불만을 표출하거나 시정을 요구하는 경우"),
        ("칭찬", "공무원, 서비스, 정책 등에 대한 긍정적인 평가를 담은 경우."),
        ("그 외", "위 유형 중 어느 것으로도 분류되지 않는 경우.")
    ]
    random.shuffle(complaint_type_items)
    complaint_type_str = "\n".join([f"- {l} : {d}" for l, d in complaint_type_items])

    emotion_items = ["긍정", "중립", "부정"]
    random.shuffle(emotion_items)
    emotion_str = ", ".join(emotion_items)

    class MinionLabel(BaseModel):
        importance:     str = Field(description="민원의 중요도. 높음/보통/낮음 중 하나")
        department:     str = Field(description="민원이 전달되어야 하는 서울시 부서. 교통실/주택실/재난안전실/시민건강국/문화본부/기후환경본부/경제실/복지실/여성가족실/분류 보류 중 하나.")
        complaint_type: str = Field(description="민원의 유형. 신고/문의/건의/칭찬/항의/그 외 중 하나")
        emotion:        str = Field(description="민원인의 감정 상태. 긍정/중립/부정 중 하나")

    parser = JsonOutputParser(pydantic_object=MinionLabel)

    return f"""SYSTEM_PROMPT = 당신은 서울시 민원 분류 담당관입니다. 지금부터 민원과 민원에 대한 답변을 읽고 키워드를 추출해주세요.
민원은 제목인 Title과 본문인 Question으로 구분되어 입력됩니다.
이때 파이썬의 Dictionary 형태로 반환하세요.

1. importance
Title과 Question을 보고 해당 민원의 중요도를 파악해 {', '.join([item[0] for item in importance_items])} 중 레이블을 구분하세요.
{importance_str}

2. department
다음은 서울시의 각 부서가 담당하는 분야입니다. 민원 내용을 보고 해당 민원이 전달되어야 할 부서를 골라주세요.

{department_str}

단, Answer를 제외한 민원(Title과 Question)을 보았을 때 다음의 경우에 해당한다면 '분류 보류'를 설정하세요.
- Title과 Question만으로 민원의 주제를 알 수 없어 특정 부서를 분류할 수 없는 경우
- 첨부 파일을 업로드 했다고 되어 있으나 Title과 Question만으로 어떤 내용인지 유추할 수 없는 경우.
- Title과 Question이 내용을 알 수 없을 정도로 짧은 경우.

3. complaint_type
Title과 Question을 보고 민원의 유형을 다음 중 하나로 구분하세요
{complaint_type_str}

4. emotion
Title과 Question을 보고 민원인의 감정상태를 {emotion_str} 중 하나로 구분하세요.

{parser.get_format_instructions()}

"""


def format_data(sample):
    return {
        "messages": [
            {"role": "system",    "content": get_system_prompt()},
            {"role": "user",      "content": f"Title: {sample['title']}\nQuestion : {sample['Question']}"},
            {"role": "assistant", "content": str(sample['assistant'])},
        ]
    }


test_dataset_list = [format_data(row) for _, row in test_df.iterrows()]
test_dataset = Dataset.from_list(test_dataset_list)
print(f"test dataset: {len(test_dataset)}개")

## 4. Tokenizer 로드 및 프롬프트 준비

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_ID)
tokenizer.padding_side = "left"

eos_token = tokenizer("<|im_end|>", add_special_tokens=False)["input_ids"][0]
print(f"eos_token id: {eos_token}")

In [ ]:
prompt_lst = []
label_lst  = []

for msgs in test_dataset["messages"]:
    text  = tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=False)
    inp   = text.split("<|im_start|>assistant")[0] + "<|im_start|>assistant"
    label = text.split("<|im_start|>assistant")[1]
    prompt_lst.append(inp)
    label_lst.append(label)

print(f"prompt: {len(prompt_lst)}개")

## 5. 평가 함수 정의

In [ ]:
KEYS = ["importance", "department", "complaint_type", "emotion"]


def parse_output(text):
    text = text.replace("<|im_end|>", "").strip()
    try:
        return json.loads(text)
    except Exception:
        pass
    try:
        result = ast.literal_eval(text)
        if isinstance(result, dict):
            return result
    except Exception:
        pass
    result = {}
    for key in KEYS:
        match = re.search(r"['\"?]" + key + r"['\"]?\s*:\s*['\"]([^'\"]+)['\"]", text)
        if match:
            result[key] = match.group(1)
    return result if result else None


def evaluate_model(pipe, prompt_lst, label_lst, model_name="Model"):
    correct    = {k: 0 for k in KEYS}
    total      = {k: 0 for k in KEYS}
    parse_fail = 0

    outputs = pipe(
        prompt_lst,
        max_new_tokens=MAX_NEW_TOKENS,
        eos_token_id=eos_token,
        do_sample=False,
        batch_size=BATCH_SIZE,
    )

    for output, prompt, label in zip(outputs, prompt_lst, label_lst):
        label_parsed = parse_output(label)
        if not label_parsed or not isinstance(label_parsed, dict):
            parse_fail += 1
            continue

        generated     = output[0]["generated_text"][len(prompt):].strip()
        output_parsed = parse_output(generated)
        if not output_parsed or not isinstance(output_parsed, dict):
            parse_fail += 1
            continue

        for key in KEYS:
            if key in label_parsed and key in output_parsed:
                total[key] += 1
                if label_parsed[key] == output_parsed[key]:
                    correct[key] += 1

    accuracy = {k: (correct[k] / total[k] * 100) if total[k] > 0 else 0.0 for k in KEYS}
    print(f"[{model_name}] 파싱 실패: {parse_fail}건")
    for k in KEYS:
        print(f"  {k}: {correct[k]}/{total[k]} = {accuracy[k]:.1f}%")
    return accuracy

## 6. 전체 Checkpoint 평가 및 저장

In [ ]:
checkpoints = sorted(
    [d for d in os.listdir(CHECKPOINT_DIR) if d.startswith("checkpoint-")],
    key=lambda x: int(x.split("-")[1])
)
print(f"발견된 checkpoint 수: {len(checkpoints)}")
print(checkpoints)

In [ ]:
all_results = {}

for ckpt in tqdm(checkpoints, desc="Evaluating checkpoints"):
    ckpt_path = os.path.join(CHECKPOINT_DIR, ckpt)
    ckpt_num  = ckpt.split("-")[1]

    fine_tuned_model = None
    pipe_ft          = None
    try:
        fine_tuned_model = AutoPeftModelForCausalLM.from_pretrained(
            ckpt_path, device_map="auto", torch_dtype=torch.float16
        )
        pipe_ft = pipeline("text-generation", model=fine_tuned_model, tokenizer=tokenizer)

        acc = evaluate_model(
            pipe_ft, prompt_lst, label_lst,
            model_name=f"{MODEL_NAME}/{ckpt}"
        )
        all_results[ckpt_num] = acc

    except Exception as e:
        print(f"Error at {ckpt}: {e}")
        all_results[ckpt_num] = None

    finally:
        if pipe_ft is not None:
            del pipe_ft
        if fine_tuned_model is not None:
            del fine_tuned_model
        torch.cuda.empty_cache()

print("\n전체 평가 완료")

In [ ]:
os.makedirs(os.path.dirname(OUTPUT_JSON), exist_ok=True)

with open(OUTPUT_JSON, "w", encoding="utf-8") as f:
    json.dump(all_results, f, ensure_ascii=False, indent=4)

print(f"저장 완료: {OUTPUT_JSON}")
print(json.dumps(all_results, ensure_ascii=False, indent=2))